# YOLO inference trong ROI — CKEY/SSH Jupyter và Google Colab

Notebook nhúng sẵn mã inference và ROI 28 camera. File `.ipynb` có thể nằm trên
laptop Arch Linux/VS Code, nhưng mọi lệnh Python và đường dẫn file thuộc về **máy
chạy kernel**. Kết nối Existing Jupyter Server qua SSH tunnel dùng GPU CKEY, không
chia sẻ tự động ổ đĩa laptop với server.

1. Kết nối kernel Python có CUDA trên máy GPU; hoặc dùng runtime Google Colab.
2. Trên CKEY, chuyển `best.pt` từ laptop tới `/workspace/tfp/best.pt` bằng SFTP/rsync,
   hoặc sửa `WEIGHTS_PATH` theo vị trí file trên server. Trên Colab, notebook vẫn hỗ
   trợ mount Drive và đường dẫn `MyDrive/tfp/models/yolo/best.pt`.
3. Đọc lần lượt 5 dataset Hugging Face; nạp `HF_TOKEN` qua môi trường hoặc nhập ẩn.
4. Checkpoint được sao lưu mỗi giờ. CKEY mặc định lưu snapshot trong
   `/workspace/tfp/checkpoints/yolo-roi-v1`; đồng bộ thư mục này về laptop để giữ
   tiến độ khi máy thuê bị xóa. Xem `docs/YOLO_INFERENCE.md` và
   `scripts/sync_yolo_checkpoint.sh` trong repo để chạy lệnh đồng bộ trên laptop.

Đầu ra là tổng xe và số xe từng class trong ROI tại từng ảnh. Camera thiếu ROI làm
dừng run. Các số đếm này chưa phải lưu lượng xe hay nhãn kẹt đường.

## 1. Cài thư viện và kiểm tra GPU

Dùng kernel PyTorch CUDA trên máy GPU thuê hoặc runtime GPU của Colab. Cell này
cài thư viện inference; không yêu cầu `google.colab` trên CKEY. Nếu notebook đang
nối kernel local của laptop, hãy chọn lại Existing Jupyter Server của máy GPU.

Cell kiểm tra import OpenCV trước khi nạp YOLO. Trên Linux Ubuntu/Debian chạy
quyền root, nếu thiếu OpenGL/GLib thì tự cài `libgl1` và gói `libglib2.0-0` hoặc
`libglib2.0-0t64` phù hợp. Nếu không có quyền root,
cell đưa ra lệnh cần quản trị viên thực hiện.


In [ ]:
import subprocess
import sys

INFERENCE_PACKAGES = ['ultralytics>=8.3,<8.4', 'Pillow>=10,<13', 'huggingface_hub>=0.26', 'pyarrow>=17', 'pandas>=2']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *INFERENCE_PACKAGES], check=True)


# Minimal GPU images may lack the OpenGL and GLib libraries used by OpenCV.
import os
import shutil


def check_opencv():
    return subprocess.run(
        [sys.executable, '-c', 'import cv2; print(cv2.__version__)'],
        capture_output=True, text=True,
    )


opencv_probe = check_opencv()
missing_system_libs = ('libGL.so.1', 'libgthread-2.0.so.0',
                       'libglib-2.0.so.0', 'libgobject-2.0.so.0')
if opencv_probe.returncode and any(name in opencv_probe.stderr for name in missing_system_libs):
    if sys.platform.startswith('linux') and os.geteuid() == 0 and shutil.which('apt-get'):
        print('OpenCV thiếu thư viện hệ thống; đang cài OpenGL và GLib...')
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        # Ubuntu 24.04+ uses the t64 package; older Debian/Ubuntu use the old name.
        glib_t64 = subprocess.run(['apt-cache', 'show', 'libglib2.0-0t64'],
                                  capture_output=True, text=True)
        glib_package = ('libglib2.0-0t64' if glib_t64.returncode == 0 and glib_t64.stdout.strip()
                        else 'libglib2.0-0')
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'libgl1', glib_package], check=True)
        opencv_probe = check_opencv()
    else:
        raise RuntimeError(
            'OpenCV thiếu thư viện hệ thống OpenGL/GLib. Trên Ubuntu/Debian, nhờ '
            'quản trị viên chạy: apt-get update && apt-get install -y libgl1 libglib2.0-0 '
            '(Ubuntu mới có thể dùng tên libglib2.0-0t64). Sau đó chạy lại cell này.'
        )
if opencv_probe.returncode:
    raise RuntimeError('Không import được OpenCV trong kernel này:\n' + opencv_probe.stderr)
print('OpenCV:', opencv_probe.stdout.strip())

import torch
assert torch.cuda.is_available(), 'Kernel hiện tại không thấy CUDA GPU. Chọn kernel trên máy GPU và kiểm tra PyTorch CUDA/driver.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))


## 2. Cấu hình theo máy chạy kernel

Notebook tự nhận diện Colab. Trên CKEY, nếu có `/workspace`, mặc định dùng
`/workspace/tfp`; nếu không, dùng thư mục làm việc của kernel. Sửa `RUNTIME_ROOT_PATH`
để đổi vị trí. Đây là đường dẫn **trên máy GPU**, không phải laptop.

- `WEIGHTS_PATH`: file `.pt` trên máy GPU. Chuyển weights từ laptop bằng SFTP/rsync;
  không cần đưa toàn bộ ảnh lên máy GPU vì chúng được đọc từ Hugging Face.
- `USE_DRIVE`: chỉ mount Drive khi thực sự chạy Colab. Máy thuê dùng `False`; nếu đã
  mount Drive bằng công cụ khác, chỉ cần điền đường dẫn mount và giữ `False`.
- `BACKUP_ENABLED = True`, `BACKUP_INTERVAL_HOURS = 1`: snapshot mỗi giờ và khi Stop.
  `BACKUP_ROOT_PATH` là nơi giữ các bản checkpoint theo `RUN_NAME`. Snapshot trên
  cùng server không bảo đảm còn lại khi server/ổ đĩa thuê bị xóa; cần volume được
  nhà cung cấp giữ lại hoặc đồng bộ về laptop.

`LIMIT = 0` chạy toàn bộ 5 dataset lần lượt; `LIMIT = 1000` giới hạn chung ảnh mới
của một lần gọi. Giữ nguyên `RUN_NAME` khi resume và đổi tên khi đổi cấu hình dự đoán.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive as _colab_drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

SOURCE_MODE = 'huggingface' #@param ['huggingface', 'local']
RUNTIME_ROOT_PATH = '/content' if IS_COLAB else ('/workspace/tfp' if Path('/workspace').is_dir() else str(Path.cwd()))
WEIGHTS_PATH = '/content/drive/MyDrive/tfp/models/yolo/best.pt' if IS_COLAB else str(Path(RUNTIME_ROOT_PATH) / 'best.pt')
LOCAL_SOURCE_PATH = str(Path(RUNTIME_ROOT_PATH) / 'snapshots')
RUN_NAME = 'yolo-roi-v1' #@param {type:'string'}
USE_DRIVE = IS_COLAB
BACKUP_ENABLED = True
BACKUP_INTERVAL_HOURS = 1 #@param {type:'number'}
BACKUP_ROOT_PATH = '/content/drive/MyDrive/tfp/yolo-roi' if IS_COLAB else str(Path(RUNTIME_ROOT_PATH) / 'checkpoints')

HF_REPOS = [
    'ericng17/hcm_traffic_snapshots',
    'ericng17/hcm_traffic_snapshots2',
    'ericng17/hcm_traffic_snapshots3',
    'ericng17/hcm_traffic_snapshots4',
    'ericng17/hcm_traffic_snapshots5',
]
VEHICLE_CLASSES = [0, 1, 2, 3]  # Xác nhận theo model.names ở bước 5.
LIMIT = 0 #@param {type:'integer'}
BATCH = 4 #@param {type:'integer'}
IMGSZ = 1024 #@param {type:'integer'}
CONF = 0.25 #@param {type:'number'}
IOU = 0.7 #@param {type:'number'}
MAX_DET = 1000 #@param {type:'integer'}
HALF = False #@param {type:'boolean'}
PREVIEW_COUNT = 28 #@param {type:'integer'}
RETRY_ERRORS = False #@param {type:'boolean'}
TIMEZONE = 'Asia/Ho_Chi_Minh' #@param {type:'string'}
HF_FORMAT = 'auto' #@param ['auto', 'raw', 'parquet']
WORKERS = 4
IMAGE_COLUMN = 'image'
PATH_COLUMN = None
CAMERA_COLUMN = None
TIMESTAMP_COLUMN = None

assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in {'.', '..'}, 'RUN_NAME chỉ là tên thư mục.'
assert LIMIT >= 0, 'LIMIT = 0 để chạy toàn bộ, hoặc số nguyên dương để chạy thử.'
assert BACKUP_INTERVAL_HOURS > 0, 'Khoảng sao lưu phải lớn hơn 0 giờ.'
RUNTIME_ROOT = Path(RUNTIME_ROOT_PATH).expanduser().resolve()
WORKDIR = RUNTIME_ROOT / 'yolo-roi-notebook'
WORKDIR.mkdir(parents=True, exist_ok=True)
WEIGHTS = Path(WEIGHTS_PATH).expanduser().resolve()
LOCAL_SOURCE = Path(LOCAL_SOURCE_PATH).expanduser().resolve()
OUTPUT = (RUNTIME_ROOT if IS_COLAB else RUNTIME_ROOT / 'outputs') / RUN_NAME
CHECKPOINT_BACKUP = Path(BACKUP_ROOT_PATH).expanduser().resolve() / RUN_NAME
print('Môi trường:', 'Google Colab' if IS_COLAB else 'Jupyter/Python trên máy GPU')
print('Nguồn:', SOURCE_MODE, '| Output:', OUTPUT)
print('Checkpoint mỗi giờ:', CHECKPOINT_BACKUP if BACKUP_ENABLED else 'tắt')
print('Giới hạn ảnh mới:', LIMIT if LIMIT else 'toàn bộ')
if SOURCE_MODE == 'huggingface':
    for index, repo in enumerate(HF_REPOS, 1):
        print(f'  {index}/{len(HF_REPOS)}: {repo}')


## 3. Kiểm tra weights, khôi phục checkpoint và nạp token

CKEY dùng file local trên server, không import `google.colab`. Colab có thể mount
Drive nếu bật `USE_DRIVE`. Nếu có snapshot của cùng run, notebook khôi phục về
output khi output chưa tồn tại. Token lấy từ `HF_TOKEN` của kernel; nếu chưa có,
Colab đọc Secrets còn Jupyter thông thường cho nhập ẩn bằng `getpass`.


In [ ]:
import json
import os
import shutil

if USE_DRIVE:
    if not IS_COLAB:
        raise RuntimeError('USE_DRIVE chỉ dành cho Colab. Trên máy thuê, đặt False và dùng đường dẫn file/mount trên server.')
    from google.colab import drive
    drive.mount('/content/drive')

if not WEIGHTS.is_file():
    raise FileNotFoundError(
        f'Không tìm thấy weights trong máy chạy kernel: {WEIGHTS}. '
        'Chuyển best.pt từ laptop lên máy GPU bằng SFTP/rsync hoặc sửa WEIGHTS_PATH. '
        'Đường dẫn Google Drive của Colab không tự tồn tại trên CKEY.')
print('Weights:', WEIGHTS)

if BACKUP_ENABLED:
    if (CHECKPOINT_BACKUP / 'counts.sqlite').is_file() and not OUTPUT.exists():
        OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(CHECKPOINT_BACKUP, OUTPUT)
        print('Đã khôi phục checkpoint:', CHECKPOINT_BACKUP)
    elif (CHECKPOINT_BACKUP / 'counts.sqlite').is_file():
        if not (OUTPUT / 'counts.sqlite').is_file():
            raise FileExistsError(f'Output đã tồn tại nhưng thiếu checkpoint: {OUTPUT}. Kiểm tra thư mục trước khi khôi phục.')
        print('Tiếp tục dùng checkpoint local:', OUTPUT)
    else:
        print('Chưa có snapshot cho run này.')

if SOURCE_MODE == 'huggingface':
    if not os.environ.get('HF_TOKEN'):
        if IS_COLAB:
            from google.colab import userdata
            try:
                token = userdata.get('HF_TOKEN')
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                token = None
        else:
            from getpass import getpass
            token = getpass('HF_TOKEN quyền đọc (Enter nếu chỉ dùng repo public): ').strip()
        if token:
            os.environ['HF_TOKEN'] = token
        del token
    print('HF_TOKEN đã có.' if os.environ.get('HF_TOKEN') else
          'Chưa có HF_TOKEN; chỉ đọc được repo public.')
else:
    assert LOCAL_SOURCE.is_dir(), f'Chưa có thư mục ảnh: {LOCAL_SOURCE}'
    print('Thư mục ảnh:', LOCAL_SOURCE)


def backup_output():
    if BACKUP_ENABLED and backup_checkpoint(OUTPUT, CHECKPOINT_BACKUP):
        print('Đã sao lưu toàn bộ output:', CHECKPOINT_BACKUP)


## 4. Nạp mã inference và ROI đã duyệt

Hai cell thu gọn sau chứa mã và ROI đã nhúng. Chạy cả hai, không cần chỉnh.
Nếu có bộ ROI mới, có thể upload JSON rồi sửa biến `ROI_PATH` trỏ đến file đó
trước bước 5; file mới phải có trạng thái `approved`.

In [ ]:
#@title Mã inference ROI (nhúng từ scripts/infer_yolo_roi.py)
"""Count selected vehicle classes inside approved camera ROIs (Python >= 3.11).

Supports local camera folders or Hugging Face raw images/embedded-image Parquet.
Run with --help. Only local detection weights are accepted; no automatic uploads.
"""
import argparse
import csv
import hashlib
import io
import json
import logging
import math
import os
import shutil
import sqlite3
import tempfile
import threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from contextlib import closing
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

from PIL import Image, ImageDraw

LOG = logging.getLogger(__name__)
EXTENSIONS = {'.jpg', '.jpeg', '.png'}
COUNT_RULE = 'bottom_center_inside_or_on_boundary_of_polygon_union'
EPS = 1e-9


def on_segment(p, a, b):
    cross = (b[0]-a[0])*(p[1]-a[1]) - (b[1]-a[1])*(p[0]-a[0])
    return (abs(cross) <= EPS and
            min(a[0], b[0])-EPS <= p[0] <= max(a[0], b[0])+EPS and
            min(a[1], b[1])-EPS <= p[1] <= max(a[1], b[1])+EPS)


def inside_polygon(point, points):
    """Ray casting for concave polygons, including edges and vertices."""
    x, y = point
    inside = False
    for a, b in zip(points, points[1:]+points[:1]):
        if on_segment(point, a, b):
            return True
        if (a[1] > y) != (b[1] > y):
            if x < a[0] + (y-a[1])*(b[0]-a[0])/(b[1]-a[1]):
                inside = not inside
    return inside


def validate_polygon(points):
    if not isinstance(points, list) or len(points) < 3:
        raise ValueError('ROI polygon requires at least three vertices')
    if any(not isinstance(p, list) or len(p) != 2 or
           any(isinstance(v, bool) or not isinstance(v, (int, float)) or
               not math.isfinite(v) or not 0 <= v <= 1 for v in p) for p in points):
        raise ValueError('ROI coordinates must be finite normalized [x,y] pairs')
    edges = list(zip(points, points[1:]+points[:1]))
    if any(a == b for a, b in edges):
        raise ValueError('ROI polygon has a zero-length edge')
    area = abs(sum(a[0]*b[1]-b[0]*a[1] for a, b in edges))/2
    if area <= .0001:
        raise ValueError('ROI polygon is degenerate')
    def orient(a, b, c):
        return (b[0]-a[0])*(c[1]-a[1]) - (b[1]-a[1])*(c[0]-a[0])
    for i, (a, b) in enumerate(edges):
        for j in range(i+1, len(edges)):
            if j == i+1 or (i == 0 and j == len(edges)-1):
                continue
            c, d = edges[j]
            if (orient(a,b,c)*orient(a,b,d) < 0 and orient(c,d,a)*orient(c,d,b) < 0
                    or on_segment(c,a,b) or on_segment(d,a,b)
                    or on_segment(a,c,d) or on_segment(b,c,d)):
                raise ValueError('ROI polygon self-intersects')


def load_rois(path):
    raw = Path(path).read_bytes()
    data = json.loads(raw)
    if data.get('schema_version') != 1 or data.get('count_rule') != COUNT_RULE:
        raise ValueError('Unsupported ROI schema or count_rule')
    if data.get('review_status') != 'approved' or data.get('missing_cameras'):
        raise ValueError('ROI file must be fully approved with no missing cameras')
    cameras = {}
    for camera in data['cameras']:
        key = camera['camera_id']
        if not isinstance(key, str) or not key or key in cameras:
            raise ValueError('ROI camera IDs must be unique nonempty strings')
        if camera.get('review_status') != 'approved' or not camera['regions']:
            raise ValueError(f'Camera {key} has no approved ROI')
        size = camera['reference_size']
        if (len(size) != 2 or any(isinstance(v, bool) or not isinstance(v, (int, float))
                                  or not math.isfinite(v) or v <= 0 for v in size)):
            raise ValueError(f'Invalid reference size: {key}')
        for region in camera['regions']:
            validate_polygon(region['points_normalized'])
        cameras[key] = camera
    if not cameras:
        raise ValueError('ROI file contains no cameras')
    return cameras, hashlib.sha256(raw).hexdigest(), data


def identity(row, path, args):
    path = Path(str(path or '').replace('\\', '/'))
    camera_keys = [args.camera_column] if args.camera_column else ['camera_id','cam_id','camera']
    time_keys = [args.timestamp_column] if args.timestamp_column else ['timestamp','datetime','time']
    camera = next((row[k] for k in camera_keys if row.get(k) is not None), path.parent.name)
    timestamp = next((row[k] for k in time_keys if row.get(k) is not None), path.stem)
    if not camera or isinstance(camera, (dict, list)):
        raise ValueError(f'Cannot identify camera for {path}')
    if isinstance(timestamp, datetime):
        dt = timestamp
    else:
        try:
            dt = datetime.strptime(str(timestamp), '%Y%m%d_%H%M%S')
        except ValueError:
            dt = datetime.fromisoformat(str(timestamp).replace('Z', '+00:00'))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=ZoneInfo(args.timezone))
    return str(camera), dt.astimezone(ZoneInfo(args.timezone)).isoformat()


def record(repo, revision, source, index, camera, timestamp, content, fingerprint=None):
    key = hashlib.sha256(json.dumps([repo, revision, source, index]).encode()).hexdigest()
    return dict(id=key, repo=repo, revision=revision, source=source, row_index=index,
                camera_id=camera, timestamp=timestamp, content=content, fingerprint=fingerprint)


def pending(db, item, args):
    if int(item['id'], 16) % args.num_shards != args.shard_index:
        return False
    old = db.execute('SELECT status,fingerprint FROM counts WHERE id=?', (item['id'],)).fetchone()
    if old and old[1] != item['fingerprint']:
        raise ValueError(f"Input changed: {item['source']}; use a new --output")
    return not old or (old[0] == 'error' and args.retry_errors)


def local_records(args, db):
    found = False
    for path in args.source.rglob('*'):
        if not path.is_file() or path.suffix.lower() not in EXTENSIONS:
            continue
        found = True
        camera, timestamp = identity({}, path.relative_to(args.source), args)
        stat = path.stat()
        item = record('', '', path.relative_to(args.source).as_posix(), 0, camera,
                      timestamp, path, f'{stat.st_size}:{stat.st_mtime_ns}')
        if pending(db, item, args):
            yield item
    if not found:
        raise ValueError('No images found under --source')


def parquet_records(local, repo, revision, source, args, db):
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(local)
    if args.image_column not in pf.schema_arrow.names:
        raise ValueError(f'{source}: missing column {args.image_column}')
    index = 0
    for batch in pf.iter_batches(batch_size=args.batch):
        for row in batch.to_pylist():
            image = row[args.image_column]
            path = row.get(args.path_column or 'path') or row.get('file_name') or row.get('filename')
            if isinstance(image, dict):
                path = path or image.get('path')
                content = image.get('bytes')
            elif isinstance(image, (bytes, type(None))):
                content = image
            else:
                raise ValueError('Parquet image must contain embedded bytes')
            camera, timestamp = identity(row, path, args)
            item = record(repo, revision, source, index, camera, timestamp, content)
            if pending(db, item, args):
                yield item
            index += 1
    # A partial final batch remains unmarked and is safely replayed on resume.
    saved = db.execute('SELECT count(*) FROM counts WHERE repo=? AND revision=? AND source=?',
                       (repo,revision,source)).fetchone()[0]
    if args.num_shards == 1 and saved == index:
        key = 'completed:'+json.dumps([repo,revision,source])
        db.execute('INSERT OR REPLACE INTO metadata VALUES (?,?)',(key,'true'))
        db.commit()


def hf_records(args, db):
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=os.environ.get('HF_TOKEN'))
    for repo in args.hf_repos:
        key = 'revision:'+repo
        old = db.execute('SELECT value FROM metadata WHERE key=?', (key,)).fetchone()
        revision = old[0] if old else api.repo_info(repo, repo_type='dataset').sha
        db.execute('INSERT OR IGNORE INTO metadata VALUES (?,?)', (key, revision))
        db.commit()
        files = sorted(api.list_repo_files(repo, repo_type='dataset', revision=revision))
        parquet = [p for p in files if p.endswith('.parquet')]
        raw = [p for p in files if Path(p).suffix.lower() in EXTENSIONS]
        use_parquet = args.format == 'parquet' or (args.format == 'auto' and bool(parquet))
        LOG.info('%s @ %s: %s', repo, revision, 'parquet' if use_parquet else 'raw')
        if not (parquet if use_parquet else raw):
            raise ValueError(f'{repo}: no files for the requested format')
        if use_parquet:
            for source in parquet:
                complete_key = 'completed:'+json.dumps([repo,revision,source])
                if not args.retry_errors and db.execute(
                        'SELECT 1 FROM metadata WHERE key=?',(complete_key,)).fetchone():
                    continue
                with tempfile.TemporaryDirectory(prefix='yolo-roi-parquet-') as tmp:
                    local = hf_hub_download(repo, source, repo_type='dataset',
                                            revision=revision, local_dir=tmp)
                    yield from parquet_records(local, repo, revision, source, args, db)
        else:
            for start in range(0, len(raw), args.batch):
                chunk = []
                for source in raw[start:start+args.batch]:
                    camera, timestamp = identity({}, source, args)
                    item = record(repo, revision, source, 0, camera, timestamp, None)
                    if pending(db, item, args):
                        chunk.append(item)
                with tempfile.TemporaryDirectory(prefix='yolo-roi-raw-') as tmp:
                    def download(item):
                        local = hf_hub_download(repo, item['source'], repo_type='dataset',
                                                revision=revision, local_dir=tmp)
                        return Path(local).read_bytes()
                    with ThreadPoolExecutor(max_workers=args.workers) as pool:
                        for item, content in zip(chunk, pool.map(download, chunk)):
                            item['content'] = content
                            yield item


def open_database(output, config):
    output.mkdir(parents=True, exist_ok=True)
    db = sqlite3.connect(output / 'counts.sqlite')
    db.execute('CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT)')
    db.execute('''CREATE TABLE IF NOT EXISTS counts (
        id TEXT PRIMARY KEY, repo TEXT, revision TEXT, source TEXT, row_index INTEGER,
        camera_id TEXT, timestamp TEXT, fingerprint TEXT, image_sha256 TEXT,
        status TEXT, total INTEGER, counts TEXT, error TEXT)''')
    encoded = json.dumps(config, sort_keys=True)
    old = db.execute("SELECT value FROM metadata WHERE key='config'").fetchone()
    if old and old[0] != encoded:
        db.close()
        raise ValueError('Configuration changed: use a new --output directory')
    db.execute("INSERT OR IGNORE INTO metadata VALUES ('config',?)", (encoded,))
    db.commit()
    (output / 'run.json').write_text(json.dumps(config, indent=2, ensure_ascii=False)+'\n')
    return db


def save_count(db, item, digest, status, counts=None, error=None):
    fields = [item[k] for k in ('id','repo','revision','source','row_index',
                                'camera_id','timestamp','fingerprint')]
    db.execute('INSERT OR REPLACE INTO counts VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)',
               fields+[digest, status, sum(counts.values()) if counts is not None else None,
                       json.dumps(counts) if counts is not None else None, error])


def export_csv(db, output, names, roi_hash):
    target = output / 'counts.csv'
    temporary = target.with_suffix('.csv.tmp')
    with temporary.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['id','repo','revision','source','row_index','camera_id','timestamp',
                         'status','total_vehicles']+[f'count_{k}_{v}' for k,v in names.items()]+
                        ['image_sha256','roi_sha256','error'])
        query = '''SELECT id,repo,revision,source,row_index,camera_id,timestamp,status,
                   total,counts,image_sha256,error FROM counts ORDER BY camera_id,timestamp,id'''
        for row in db.execute(query):
            counts = json.loads(row[9]) if row[9] else {}
            writer.writerow(list(row[:9])+[counts.get(str(k),0) if row[7]=='ok' else ''
                                           for k in names]+[row[10],roi_hash,row[11]])
    temporary.replace(target)


def backup_checkpoint(output, destination):
    """Snapshot committed SQLite rows, including while inference is running.

    SQLite's backup API reads a consistent snapshot through a separate connection.
    Stage destination files before replacement so an interrupted copy never
    truncates the previous checkpoint database. One backup writer per destination.
    """
    output, destination = Path(output), Path(destination)
    if not (output / 'counts.sqlite').is_file():
        return False
    if (output.resolve().is_relative_to(destination.resolve()) or
            destination.resolve().is_relative_to(output.resolve())):
        raise ValueError('Checkpoint destination must be separate from the output directory')
    config = json.loads((output / 'run.json').read_text())
    previous_config = destination / 'run.json'
    if previous_config.is_file() and json.loads(previous_config.read_text()) != config:
        raise ValueError('Backup configuration differs; choose another run/destination')
    with tempfile.TemporaryDirectory(prefix='yolo-roi-checkpoint-') as tmp:
        snapshot = Path(tmp)
        source_uri = (output / 'counts.sqlite').resolve().as_uri()+'?mode=ro'
        with closing(sqlite3.connect(source_uri, uri=True, timeout=60)) as source:
            with closing(sqlite3.connect(snapshot / 'counts.sqlite')) as target:
                source.backup(target)
                saved = target.execute("SELECT value FROM metadata WHERE key='config'").fetchone()
                if not saved or json.loads(saved[0]) != config:
                    raise ValueError('Database and run.json configurations differ')
                export_csv(target, snapshot,
                           {int(k): v for k,v in config['names'].items()}, config['roi_sha256'])
        for filename in ('run.json','roi.json'):
            shutil.copy2(output / filename, snapshot / filename)
        destination.mkdir(parents=True, exist_ok=True)

        def publish(source_path, target_path):
            # Same destination filesystem for replace, including mounted Drive.
            with tempfile.NamedTemporaryFile(dir=target_path.parent, prefix='.'+target_path.name,
                                             suffix='.tmp', delete=False) as f:
                staged = Path(f.name)
            try:
                shutil.copy2(source_path, staged)
                staged.replace(target_path)
            finally:
                staged.unlink(missing_ok=True)

        for filename in ('run.json','roi.json','counts.csv','counts.sqlite'):
            publish(snapshot / filename, destination / filename)
        for preview in (output / 'previews').glob('*.jpg'):
            target_dir = destination / 'previews'
            target_dir.mkdir(exist_ok=True)
            publish(preview, target_dir / preview.name)
    LOG.info('Checkpoint backed up: %s', destination)
    return True


class PeriodicCheckpoint:
    """Back up on a background thread; the inference connection stays on its thread."""

    def __init__(self, output, destination, interval_seconds=3600):
        if not math.isfinite(interval_seconds) or interval_seconds <= 0:
            raise ValueError('Checkpoint interval must be positive and finite')
        self.output = Path(output)
        self.destination = Path(destination)
        self.interval_seconds = interval_seconds
        self._stop = threading.Event()
        self._thread = None

    def _worker(self):
        while not self._stop.wait(self.interval_seconds):
            try:
                backup_checkpoint(self.output, self.destination)
            except Exception:
                LOG.exception('Periodic checkpoint failed; retrying at the next interval')

    def start(self):
        if self._thread is not None:
            raise RuntimeError('Periodic checkpoint worker already started')
        self._thread = threading.Thread(target=self._worker, name='roi-checkpoint', daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        if self._thread is not None:
            # Wait for any active backup before the caller makes a final backup.
            self._thread.join()


def filter_boxes(result, camera, class_ids):
    if result.boxes is None:
        raise RuntimeError('Expected detection boxes')
    height, width = result.orig_shape
    rw, rh = camera['reference_size']
    if width <= 0 or height <= 0 or abs(width*rh/(height*rw)-1) > .01:
        raise ValueError(f"Image aspect ratio differs from ROI reference: {camera['camera_id']}")
    boxes = result.boxes.xyxy.cpu().tolist()
    classes = result.boxes.cls.cpu().tolist()
    if len(boxes) != len(classes):
        raise RuntimeError('Detection coordinates/classes mismatch')
    kept, counts = [], Counter()
    for box, class_id in zip(boxes, classes):
        class_id = int(class_id)
        if class_id not in class_ids:
            continue
        if len(box) != 4 or not all(math.isfinite(v) for v in box):
            raise RuntimeError('Invalid detection coordinates')
        x1, y1, x2, y2 = box
        point = ((x1+x2)/(2*width), y2/height)
        if any(inside_polygon(point, r['points_normalized']) for r in camera['regions']):
            counts[class_id] += 1
            kept.append((box, class_id))
    return counts, kept


def save_preview(image, camera, kept, counts, names, target):
    image = image.copy()
    draw = ImageDraw.Draw(image)
    for region in camera['regions']:
        points = [(x*image.width,y*image.height) for x,y in region['points_normalized']]
        draw.line(points+points[:1], fill='#36e8ae', width=2)
    for box, class_id in kept:
        draw.rectangle(box, outline='#ffca62', width=2)
        x1,y1,x2,y2 = box
        x,y = (x1+x2)/2,y2
        draw.ellipse((x-2,y-2,x+2,y+2), fill='#ffca62')
        draw.text((x1,y1), str(class_id), fill='#ffca62')
    label = f'ROI total: {sum(counts.values())} | '+', '.join(
        f'{k} {v}: {counts[k]}' for k,v in names.items())
    draw.rectangle((0,0,image.width,30), fill='black')
    draw.text((4,4), label, fill='white')
    image.save(target, quality=95)


def process_batch(model, records, db, args, rois, names, device):
    images, good = [], []
    for item in records:
        camera = rois.get(item['camera_id'])
        if camera is None:
            raise ValueError(f"No approved ROI for camera {item['camera_id']}")
        try:
            content = item['content']
            if isinstance(content, Path):
                stat = content.stat()
                if f'{stat.st_size}:{stat.st_mtime_ns}' != item['fingerprint']:
                    raise RuntimeError(f'Input changed while reading: {content}')
                content = content.read_bytes()
            if not content:
                raise ValueError('Missing embedded image bytes')
            digest = hashlib.sha256(content).hexdigest()
            with Image.open(io.BytesIO(content)) as image:
                image.load()
                images.append(image.convert('RGB'))
            good.append((item, digest))
        except (OSError, ValueError) as exc:
            save_count(db, item, None, 'error', error=f'{type(exc).__name__}: {exc}')
    if images:
        results = list(model.predict(source=images, device=device, imgsz=args.imgsz,
            conf=args.conf, iou=args.iou, max_det=args.max_det, classes=list(names),
            half=args.half, stream=True, verbose=False, save=False))
        if len(results) != len(good):
            raise RuntimeError('YOLO result count differs from input count')
        preview_dir = args.output / 'previews'
        preview_dir.mkdir(exist_ok=True)
        for image, result, (item,digest) in zip(images, results, good):
            if tuple(result.orig_shape) != (image.height,image.width):
                raise RuntimeError('YOLO original shape differs from source image')
            counts, kept = filter_boxes(result, rois[item['camera_id']], names)
            save_count(db, item, digest, 'ok', counts)
            if args.previews_saved < args.preview_count:
                save_preview(image, rois[item['camera_id']], kept, counts, names,
                             preview_dir / f"{item['id']}.jpg")
                args.previews_saved += 1
    db.commit()


def parser():
    p = argparse.ArgumentParser(description=__doc__)
    source = p.add_mutually_exclusive_group(required=True)
    source.add_argument('--source', type=Path, help='Local image directory with camera subfolders')
    source.add_argument('--hf-repos', nargs='+', help='Hugging Face dataset repo IDs')
    p.add_argument('--roi', type=Path, required=True, help='Approved camera ROI JSON')
    p.add_argument('--weights', type=Path, required=True, help='Local detection .pt weights')
    p.add_argument('--classes', type=int, nargs='+', required=True, help='Vehicle IDs from model.names')
    p.add_argument('--output', type=Path, required=True)
    p.add_argument('--device', default='auto', help='auto requires CUDA; use cpu explicitly for CPU')
    p.add_argument('--batch', type=int, default=4)
    p.add_argument('--imgsz', type=int, default=1024)
    p.add_argument('--conf', type=float, default=.25)
    p.add_argument('--iou', type=float, default=.7)
    p.add_argument('--max-det', type=int, default=1000)
    p.add_argument('--half', action='store_true')
    p.add_argument('--timezone', default='Asia/Ho_Chi_Minh')
    p.add_argument('--limit', type=int, help='Maximum new/retried images; omit for all')
    p.add_argument('--preview-count', type=int, default=28)
    p.add_argument('--retry-errors', action='store_true')
    p.add_argument('--num-shards', type=int, default=1)
    p.add_argument('--shard-index', type=int, default=0)
    p.add_argument('--format', choices=['auto','raw','parquet'], default='auto', help='HF format')
    p.add_argument('--workers', type=int, default=4, help='HF raw download workers')
    p.add_argument('--image-column', default='image')
    p.add_argument('--path-column')
    p.add_argument('--camera-column')
    p.add_argument('--timestamp-column')
    return p


def run(args):
    if any(v < 1 for v in (args.batch,args.imgsz,args.max_det,args.workers,args.num_shards)):
        raise ValueError('batch, imgsz, max-det, workers and num-shards must be positive')
    if (not 0 <= args.conf <= 1 or not 0 <= args.iou <= 1 or args.preview_count < 0
            or not 0 <= args.shard_index < args.num_shards
            or (args.limit is not None and args.limit < 1)):
        raise ValueError('Invalid threshold, preview count, shard index or limit')
    if not args.weights.is_file() or (args.source is not None and not args.source.is_dir()):
        raise ValueError('Local weights or source directory missing')
    if args.hf_repos and len(set(args.hf_repos)) != len(args.hf_repos):
        raise ValueError('Duplicate HF repos')
    ZoneInfo(args.timezone)
    rois, roi_hash, roi_data = load_rois(args.roi)
    import torch
    import ultralytics
    from ultralytics import YOLO
    device = args.device
    if device == 'auto':
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA unavailable: enable GPU or pass --device cpu')
        device = '0'
    if args.half and (device == 'cpu' or device == 'mps'):
        raise ValueError('--half requires CUDA')
    model = YOLO(str(args.weights))
    if model.task != 'detect':
        raise ValueError('Detection weights required')
    all_names = {int(k):v for k,v in model.names.items()}
    if not set(args.classes) <= all_names.keys():
        raise ValueError(f'Invalid vehicle classes; model mapping: {all_names}')
    names = {k:all_names[k] for k in sorted(set(args.classes))}
    with args.weights.open('rb') as f:
        weights_hash = hashlib.file_digest(f, 'sha256').hexdigest()
    config = dict(engine_version=1, weights_sha256=weights_hash, roi_sha256=roi_hash,
                  count_rule=COUNT_RULE, names=names,
                  source=str(args.source.resolve()) if args.source else None,
                  hf_repos=args.hf_repos, imgsz=args.imgsz, conf=args.conf, iou=args.iou,
                  max_det=args.max_det, half=args.half, timezone=args.timezone,
                  num_shards=args.num_shards, shard_index=args.shard_index, format=args.format,
                  image_column=args.image_column, path_column=args.path_column,
                  camera_column=args.camera_column, timestamp_column=args.timestamp_column,
                  torch=torch.__version__, ultralytics=ultralytics.__version__)
    db = open_database(args.output, config)
    (args.output / 'roi.json').write_text(json.dumps(roi_data,indent=2,ensure_ascii=False)+'\n')
    args.previews_saved = sum(1 for _ in (args.output / 'previews').glob('*.jpg'))
    records = local_records(args,db) if args.source else hf_records(args,db)
    processed, batch = 0, []
    LOG.info('Vehicle classes: %s; ROI cameras: %d; device: %s', names,len(rois),device)
    try:
        for item in records:
            if item['camera_id'] not in rois:
                raise ValueError(f"No approved ROI for camera {item['camera_id']}")
            batch.append(item)
            processed += 1
            if len(batch) == args.batch:
                process_batch(model,batch,db,args,rois,names,device)
                batch.clear()
                LOG.info('Processed %d new/retried images', processed)
            if args.limit and processed >= args.limit:
                break
        if batch:
            process_batch(model,batch,db,args,rois,names,device)
    finally:
        records.close()
        db.rollback()
        export_csv(db,args.output,names,roi_hash)
        LOG.info('Saved %s: %s',args.output,db.execute(
            'SELECT status,count(*) FROM counts GROUP BY status').fetchall())
        db.close()


In [ ]:
#@title ROI đã duyệt — 28 camera
ROI_JSON = r'''{
  "schema_version": 1,
  "created_on": "2026-11-15",
  "review_status": "approved",
  "count_rule": "bottom_center_inside_or_on_boundary_of_polygon_union",
  "coordinate_system": "normalized_xy: x/image_width, y/image_height; original image orientation",
  "camera_node_aggregation": "union_of_regions; each detection counted at most once",
  "direction_policy": "regions describe image geometry only; no legal travel direction inferred",
  "reference_limitation": "One reference snapshot per camera: 27 night snapshots and one daytime snapshot for Hoang Van Thu–Ut Tich. Day/night and viewpoint stability validation pending.",
  "cameras": [
    {
      "index": 1,
      "camera_id": "56df807bc062921100c143da",
      "name": "Truong_Chinh_Au_Co",
      "reference_image": "data/raw/snapshots/56df807bc062921100c143da/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "e00cec19fadc2dd4ef7c1d8200c9187faee1d69bca1641984b526dfda602da69",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "junction",
          "points_normalized": [
            [
              0.007812,
              0.475694
            ],
            [
              0.226562,
              0.34375
            ],
            [
              0.478571,
              0.119224
            ],
            [
              0.546875,
              0.118056
            ],
            [
              0.6875,
              0.118056
            ],
            [
              0.695312,
              0.315972
            ],
            [
              0.779297,
              0.413194
            ],
            [
              0.990234,
              0.520833
            ],
            [
              1,
              1
            ],
            [
              0.007812,
              0.986111
            ]
          ],
          "points_px": [
            [
              4,
              137
            ],
            [
              116,
              99
            ],
            [
              245.03,
              34.34
            ],
            [
              280,
              34
            ],
            [
              352,
              34
            ],
            [
              356,
              91
            ],
            [
              399,
              119
            ],
            [
              507,
              150
            ],
            [
              512,
              288
            ],
            [
              4,
              284
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        }
      ],
      "note": "Giao lộ rộng; ROI gộp xe trong nút giao và phần tiếp cận đủ rõ, không đại diện một hướng lưu thông duy nhất. Cần ảnh ban ngày kiểm tra ranh vỉa hè hai bên."
    },
    {
      "index": 2,
      "camera_id": "586e1d4ef9fab7001111b0a2",
      "name": "Cong_Hoa_Duong_C18",
      "reference_image": "data/raw/snapshots/586e1d4ef9fab7001111b0a2/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "3b11f036ca33ccaf40c96265d44402220d880f45880176827acebb431165774f",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "carriageway_left",
          "points_normalized": [
            [
              0.151587,
              0.198236
            ],
            [
              0.249206,
              0.196825
            ],
            [
              0.6875,
              0.986111
            ],
            [
              0.076172,
              0.986111
            ]
          ],
          "points_px": [
            [
              77.61,
              57.09
            ],
            [
              127.59,
              56.69
            ],
            [
              352,
              284
            ],
            [
              39,
              284
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        },
        {
          "region_id": "R2",
          "name": "carriageway_right",
          "points_normalized": [
            [
              0.255556,
              0.199647
            ],
            [
              0.377778,
              0.189771
            ],
            [
              0.990234,
              0.625
            ],
            [
              0.990234,
              0.986111
            ],
            [
              0.744141,
              0.986111
            ],
            [
              0.421875,
              0.482639
            ]
          ],
          "points_px": [
            [
              130.84,
              57.5
            ],
            [
              193.42,
              54.65
            ],
            [
              507,
              180
            ],
            [
              507,
              284
            ],
            [
              381,
              284
            ],
            [
              216,
              139
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        }
      ],
      "note": "Tách hai phần đường qua dải phân cách; loại vùng xa và vỉa hè. Cạnh trên chỉ là ngưỡng quan sát dự thảo, cần xem thêm ảnh ngày/đêm."
    },
    {
      "index": 3,
      "camera_id": "586e1f18f9fab7001111b0a5",
      "name": "Cong_Hoa_Truong_Chinh",
      "reference_image": "data/raw/snapshots/586e1f18f9fab7001111b0a5/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "359b5454b8bd1a381b6f7ab3d36cd5cc36ffb04e92da9d6ee4b023cd07fdb5f2",
      "review_status": "approved",
      "confidence": "low",
      "regions": [
        {
          "region_id": "R1",
          "name": "carriageway_left",
          "points_normalized": [
            [
              0.005859,
              0.263889
            ],
            [
              0.164286,
              0.202469
            ],
            [
              0.652381,
              0.977072
            ],
            [
              0.007812,
              0.986111
            ]
          ],
          "points_px": [
            [
              3,
              76
            ],
            [
              84.11,
              58.31
            ],
            [
              334.02,
              281.4
            ],
            [
              4,
              284
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        },
        {
          "region_id": "R2",
          "name": "carriageway_right",
          "points_normalized": [
            [
              0.169841,
              0.20388
            ],
            [
              0.229365,
              0.162963
            ],
            [
              0.709524,
              0.241975
            ],
            [
              0.992857,
              0.35485
            ],
            [
              0.990234,
              0.986111
            ],
            [
              0.720703,
              0.986111
            ],
            [
              0.483333,
              0.553792
            ]
          ],
          "points_px": [
            [
              86.96,
              58.72
            ],
            [
              117.43,
              46.93
            ],
            [
              363.28,
              69.69
            ],
            [
              508.34,
              102.2
            ],
            [
              507,
              284
            ],
            [
              369,
              284
            ],
            [
              247.47,
              159.49
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        }
      ],
      "note": "Ảnh đêm tối và lóa đèn, ranh phải khó thấy. Hai ROI nằm hai phía dải phân cách; cần xác nhận lại bằng ảnh sáng trước khi duyệt."
    },
    {
      "index": 4,
      "camera_id": "586e25e1f9fab7001111b0ae",
      "name": "Truong_Chinh_Tan_Ky_Tan_Quy",
      "reference_image": "data/raw/snapshots/586e25e1f9fab7001111b0ae/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "3638834c95363b9d936ba215728fbf1c575a441271f17db03aca51d8ecdaef02",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "junction",
          "points_normalized": [
            [
              0.005859,
              0.399306
            ],
            [
              0,
              0.254674
            ],
            [
              0.273438,
              0.315972
            ],
            [
              0.334127,
              0.337919
            ],
            [
              0.431746,
              0.236332
            ],
            [
              0.48254,
              0.213757
            ],
            [
              0.629365,
              0.256085
            ],
            [
              0.794444,
              0.302646
            ],
            [
              0.769531,
              0.524306
            ],
            [
              0.861328,
              0.659722
            ],
            [
              0.990234,
              0.767361
            ],
            [
              0.990234,
              0.986111
            ],
            [
              0.005859,
              0.986111
            ]
          ],
          "points_px": [
            [
              3,
              115
            ],
            [
              0,
              73.35
            ],
            [
              140,
              91
            ],
            [
              171.07,
              97.32
            ],
            [
              221.05,
              68.06
            ],
            [
              247.06,
              61.56
            ],
            [
              322.23,
              73.75
            ],
            [
              406.76,
              87.16
            ],
            [
              394,
              151
            ],
            [
              441,
              190
            ],
            [
              507,
              221
            ],
            [
              507,
              284
            ],
            [
              3,
              284
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        }
      ],
      "note": "ROI nút giao gồm xe đi ngang/chuyển hướng. Loại mặt tiền cửa hàng và nhánh xa; cây/dây điện che một phần phía phải. Chưa gán hướng graph."
    },
    {
      "index": 5,
      "camera_id": "5874769eb807da0011e33cec",
      "name": "Cong_Hoa_Ut_Tich2",
      "reference_image": "data/raw/snapshots/5874769eb807da0011e33cec/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "f5c7b9be05bb92a64ff041795ce8cac9875f024f5afb37941fa46ccaccd466dd",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "visible_road",
          "points_normalized": [
            [
              0.574219,
              0.232639
            ],
            [
              0.75,
              0.225694
            ],
            [
              0.783203,
              0.986111
            ],
            [
              0.021484,
              0.986111
            ],
            [
              0.267578,
              0.638889
            ],
            [
              0.488281,
              0.399306
            ]
          ],
          "points_px": [
            [
              294,
              67
            ],
            [
              384,
              65
            ],
            [
              401,
              284
            ],
            [
              11,
              284
            ],
            [
              137,
              184
            ],
            [
              250,
              115
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        }
      ],
      "note": "Đường hẹp nhìn chéo, một ROI bao phần đường đủ rõ và cả hai phía vạch giữa; không suy ra một chiều từ ảnh. Xe cắt biên dưới tính theo điểm đáy giữa box."
    },
    {
      "index": 6,
      "camera_id": "587476e3b807da0011e33cee",
      "name": "Cong_Hoa_Ut_Tich1",
      "reference_image": "data/raw/snapshots/587476e3b807da0011e33cee/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "7509b5ad462a88bb78b73c5fbf8d6adda7d92a5e3700036459e584cf0a11d467",
      "review_status": "approved",
      "confidence": "low",
      "regions": [
        {
          "region_id": "R1",
          "name": "road_left_of_island",
          "points_normalized": [
            [
              0.005859,
              0.819444
            ],
            [
              0.201172,
              0.708333
            ],
            [
              0.257937,
              0.556614
            ],
            [
              0.453175,
              0.529806
            ],
            [
              0.643651,
              0.47478
            ],
            [
              0.693651,
              0.605997
            ],
            [
              0.462698,
              0.771076
            ],
            [
              0.361111,
              0.854321
            ],
            [
              0.183594,
              0.986111
            ],
            [
              0.005859,
              0.986111
            ]
          ],
          "points_px": [
            [
              3,
              236
            ],
            [
              103,
              204
            ],
            [
              132.06,
              160.3
            ],
            [
              232.03,
              152.58
            ],
            [
              329.55,
              136.74
            ],
            [
              355.15,
              174.53
            ],
            [
              236.9,
              222.07
            ],
            [
              184.89,
              246.04
            ],
            [
              94,
              284
            ],
            [
              3,
              284
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        },
        {
          "region_id": "R2",
          "name": "road_right_of_island",
          "points_normalized": [
            [
              0.659524,
              0.473369
            ],
            [
              0.896484,
              0.506944
            ],
            [
              1,
              0.529806
            ],
            [
              1,
              0.814815
            ],
            [
              1,
              1
            ],
            [
              0.367188,
              0.986111
            ],
            [
              0.607422,
              0.770833
            ],
            [
              0.708984,
              0.642361
            ]
          ],
          "points_px": [
            [
              337.68,
              136.33
            ],
            [
              459,
              146
            ],
            [
              512,
              152.58
            ],
            [
              512,
              234.67
            ],
            [
              512,
              288
            ],
            [
              188,
              284
            ],
            [
              311,
              222
            ],
            [
              363,
              185
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        }
      ],
      "note": "Tách hai vùng đường hai bên đảo cây. Dây điện, biển báo/cột và tán cây che một phần ảnh; ranh phức tạp, cần ảnh ban ngày và người dùng xác nhận."
    },
    {
      "index": 7,
      "camera_id": "58747923b807da0011e33cf5",
      "name": "Nut_Giao_Bay_Hien2",
      "reference_image": "data/raw/snapshots/58747923b807da0011e33cf5/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "35334c2f0e8be40da3edc79eaee2a7f1a88b61551f36051f0970a10e0fe9a1e2",
      "review_status": "approved",
      "confidence": "low",
      "regions": [
        {
          "region_id": "R1",
          "name": "near_approach",
          "points_normalized": [
            [
              0.003968,
              0.251852
            ],
            [
              0.076984,
              0.236332
            ],
            [
              0.149206,
              0.244797
            ],
            [
              0.234127,
              0.313933
            ],
            [
              0.349206,
              0.525573
            ],
            [
              0.446032,
              0.703351
            ],
            [
              0.597619,
              0.998236
            ],
            [
              0,
              0.999647
            ]
          ],
          "points_px": [
            [
              2.03,
              72.53
            ],
            [
              39.42,
              68.06
            ],
            [
              76.39,
              70.5
            ],
            [
              119.87,
              90.41
            ],
            [
              178.79,
              151.37
            ],
            [
              228.37,
              202.57
            ],
            [
              305.98,
              287.49
            ],
            [
              0,
              287.9
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        },
        {
          "region_id": "R2",
          "name": "road_right_of_barrier",
          "points_normalized": [
            [
              0.234127,
              0.281482
            ],
            [
              0.361905,
              0.251852
            ],
            [
              0.988281,
              0.347222
            ],
            [
              0.990234,
              0.986111
            ],
            [
              0.606349,
              0.982716
            ]
          ],
          "points_px": [
            [
              119.87,
              81.07
            ],
            [
              185.3,
              72.53
            ],
            [
              506,
              100
            ],
            [
              507,
              284
            ],
            [
              310.45,
              283.02
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        }
      ],
      "note": "Tách tiếp cận gần và vùng đường phải rào; bỏ đảo giao thông và nhánh xa chói sáng. Kết nối hình học không khẳng định hướng đi thực tế."
    },
    {
      "index": 8,
      "camera_id": "587c8183b807da0011e33d3f",
      "name": "Hoang_Van_Thu_Tran_Quoc_Hoan",
      "reference_image": "data/raw/snapshots/587c8183b807da0011e33d3f/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "700647227458a89879ef98aef4fbbf2dc2a09770c7ba144db51b1093065c8ef9",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "junction_excluding_island",
          "points_normalized": [
            [
              0,
              0.385891
            ],
            [
              0.155556,
              0.344974
            ],
            [
              0.357422,
              0.524306
            ],
            [
              0.53125,
              0.364583
            ],
            [
              0.71875,
              0.34375
            ],
            [
              0.810317,
              0.392945
            ],
            [
              0.844444,
              0.631393
            ],
            [
              0.87619,
              0.797884
            ],
            [
              0.940476,
              1
            ],
            [
              0.007812,
              0.986111
            ],
            [
              0.004762,
              0.806349
            ],
            [
              0.053968,
              0.665256
            ],
            [
              0.051587,
              0.536861
            ]
          ],
          "points_px": [
            [
              0,
              111.14
            ],
            [
              79.64,
              99.35
            ],
            [
              183,
              151
            ],
            [
              272,
              105
            ],
            [
              368,
              99
            ],
            [
              414.88,
              113.17
            ],
            [
              432.36,
              181.84
            ],
            [
              448.61,
              229.79
            ],
            [
              481.52,
              288
            ],
            [
              4,
              284
            ],
            [
              2.44,
              232.23
            ],
            [
              27.63,
              191.59
            ],
            [
              26.41,
              154.62
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        }
      ],
      "note": "Một đa giác lõm loại đảo cây tam giác; gộp phần giao lộ gần, không gán hướng. Tán cây che mép trái/phải nên ROI giữ vào trong phần đường nhìn thấy."
    },
    {
      "index": 9,
      "camera_id": "587c8661b807da0011e33d43",
      "name": "Le_Van_Si_Hoang_Van_Thu",
      "reference_image": "data/raw/snapshots/587c8661b807da0011e33d43/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "94eba155ef94cd53397f87e759c50283b4c4559b3c73519982ff5c43000684a1",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "junction",
          "points_normalized": [
            [
              0.115079,
              0.160141
            ],
            [
              0.298828,
              0.222222
            ],
            [
              0.610317,
              0.186949
            ],
            [
              0.757937,
              0.253263
            ],
            [
              0.938095,
              0.285714
            ],
            [
              0.999206,
              0.3097
            ],
            [
              0.990234,
              0.986111
            ],
            [
              0.007812,
              0.986111
            ],
            [
              0.007812,
              0.472222
            ],
            [
              0.111328,
              0.385417
            ]
          ],
          "points_px": [
            [
              58.92,
              46.12
            ],
            [
              153,
              64
            ],
            [
              312.48,
              53.84
            ],
            [
              388.06,
              72.94
            ],
            [
              480.3,
              82.29
            ],
            [
              511.59,
              89.19
            ],
            [
              507,
              284
            ],
            [
              4,
              284
            ],
            [
              4,
              136
            ],
            [
              57,
              111
            ]
          ],
          "direction_note": "Vùng hình học dự thảo; chưa xác nhận hướng lưu thông từ một ảnh đêm."
        }
      ],
      "note": "ROI giao lộ gần bao cả xe đi thẳng và rẽ; loại mặt tiền/vỉa hè và đường quá xa. Nếu graph cần một hướng thì phải tách lại ROI."
    },
    {
      "index": 10,
      "camera_id": "58ad6214bd82540010390be2",
      "name": "Cong_Hoa_Hoang_Hoa_Tham",
      "reference_image": "data/raw/snapshots/58ad6214bd82540010390be2/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "9bd92a3aac63e5943592a8842121ca17e967decab7d39f6a101eec94236e7c4c",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "carriageway",
          "points_normalized": [
            [
              0.130859,
              1
            ],
            [
              0.191406,
              0.659722
            ],
            [
              0.2,
              0.501587
            ],
            [
              0.220635,
              0.27866
            ],
            [
              0.292969,
              0.267361
            ],
            [
              0.359375,
              0.260417
            ],
            [
              0.404297,
              0.413194
            ],
            [
              0.496094,
              0.666667
            ],
            [
              0.617188,
              1
            ]
          ],
          "points_px": [
            [
              67,
              288
            ],
            [
              98,
              190
            ],
            [
              102.4,
              144.46
            ],
            [
              112.97,
              80.25
            ],
            [
              150,
              77
            ],
            [
              184,
              75
            ],
            [
              207,
              119
            ],
            [
              254,
              192
            ],
            [
              316,
              288
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        }
      ],
      "note": "Phần đường gần và trung cảnh; loại vỉa hè phải và cây bên trái. Một vùng gộp, chưa xác nhận chiều xe theo biển báo."
    },
    {
      "index": 11,
      "camera_id": "59414c883dcfc400106f260b",
      "name": "Cong_Hoa_Binh_Gia",
      "reference_image": "data/raw/snapshots/59414c883dcfc400106f260b/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "22f50e004812877aa8d894d0433ed7e674a6b0302b780f4c45e4030cffc8f0b9",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "near_carriageway",
          "points_normalized": [
            [
              0,
              0.555556
            ],
            [
              0.263672,
              0.423611
            ],
            [
              0.523438,
              0.288194
            ],
            [
              0.734375,
              0.15625
            ],
            [
              0.759766,
              0.15625
            ],
            [
              0.542969,
              1
            ],
            [
              0,
              1
            ]
          ],
          "points_px": [
            [
              0,
              160
            ],
            [
              135,
              122
            ],
            [
              268,
              83
            ],
            [
              376,
              45
            ],
            [
              389,
              45
            ],
            [
              278,
              288
            ],
            [
              0,
              288
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        },
        {
          "region_id": "R2",
          "name": "far_side_carriageway",
          "points_normalized": [
            [
              0.783203,
              0.166667
            ],
            [
              0.814453,
              0.208333
            ],
            [
              0.857937,
              0.514286
            ],
            [
              0.915873,
              1
            ],
            [
              0.553175,
              1
            ]
          ],
          "points_px": [
            [
              401,
              48
            ],
            [
              417,
              60
            ],
            [
              439.26,
              148.11
            ],
            [
              468.93,
              288
            ],
            [
              283.23,
              288
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        }
      ],
      "note": "Hai phía dải phân cách. Vùng phải dừng trước thiết bị camera/cột che tiền cảnh; cần ảnh ngày để kiểm tra mép vỉa hè và hướng đường."
    },
    {
      "index": 12,
      "camera_id": "5d8cdb0a766c880017188964",
      "name": "Truong_Chinh_Xuan_Hong",
      "reference_image": "data/raw/snapshots/5d8cdb0a766c880017188964/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "7311bcd39ba30ce5e566e432a1780fefb09adfe0fae34d737fd1c6ada94db0c2",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "left_of_barrier",
          "points_normalized": [
            [
              0,
              0.347222
            ],
            [
              0.060547,
              0.263889
            ],
            [
              0.150794,
              0.247619
            ],
            [
              0.299206,
              0.199647
            ],
            [
              0.484375,
              0.305556
            ],
            [
              0.480469,
              0.506944
            ],
            [
              0.410156,
              0.673611
            ],
            [
              0.353516,
              0.8125
            ],
            [
              0.205078,
              1
            ],
            [
              0,
              1
            ]
          ],
          "points_px": [
            [
              0,
              100
            ],
            [
              31,
              76
            ],
            [
              77.21,
              71.31
            ],
            [
              153.19,
              57.5
            ],
            [
              248,
              88
            ],
            [
              246,
              146
            ],
            [
              210,
              194
            ],
            [
              181,
              234
            ],
            [
              105,
              288
            ],
            [
              0,
              288
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        },
        {
          "region_id": "R2",
          "name": "right_of_barrier",
          "points_normalized": [
            [
              0.523016,
              0.270194
            ],
            [
              0.62619,
              0.273016
            ],
            [
              0.657143,
              0.402822
            ],
            [
              0.779365,
              0.463492
            ],
            [
              1,
              0.497355
            ],
            [
              1,
              1
            ],
            [
              0.242188,
              1
            ],
            [
              0.388672,
              0.819444
            ],
            [
              0.445312,
              0.65625
            ],
            [
              0.503906,
              0.517361
            ]
          ],
          "points_px": [
            [
              267.78,
              77.82
            ],
            [
              320.61,
              78.63
            ],
            [
              336.46,
              116.01
            ],
            [
              399.03,
              133.49
            ],
            [
              512,
              143.24
            ],
            [
              512,
              288
            ],
            [
              124,
              288
            ],
            [
              199,
              236
            ],
            [
              228,
              189
            ],
            [
              258,
              149
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        }
      ],
      "note": "Chia hai phía rào chắn công trường, loại đường chân trời lóa và vùng vỉa hè bên phải trên ảnh. Không suy ra chiều lưu thông từ một frame."
    },
    {
      "index": 13,
      "camera_id": "5d8cdbdc766c88001718896a",
      "name": "Hoang_Van_Thu_Cong_Hoa",
      "reference_image": "data/raw/snapshots/5d8cdbdc766c88001718896a/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "0d8b003a9211dd14b94f8767e9ae196700ed04b880b9e048b151e27b996c8b99",
      "review_status": "approved",
      "confidence": "low",
      "regions": [
        {
          "region_id": "R1",
          "name": "foreground_turning_area",
          "points_normalized": [
            [
              0,
              0.326631
            ],
            [
              0.099206,
              0.349206
            ],
            [
              0.339844,
              0.513889
            ],
            [
              0.447266,
              0.576389
            ],
            [
              0.429688,
              0.649306
            ],
            [
              0.462891,
              0.753472
            ],
            [
              0.564453,
              0.829861
            ],
            [
              0.765625,
              0.927083
            ],
            [
              0.851562,
              1
            ],
            [
              0,
              1
            ]
          ],
          "points_px": [
            [
              0,
              94.07
            ],
            [
              50.79,
              100.57
            ],
            [
              174,
              148
            ],
            [
              229,
              166
            ],
            [
              220,
              187
            ],
            [
              237,
              217
            ],
            [
              289,
              239
            ],
            [
              392,
              267
            ],
            [
              436,
              288
            ],
            [
              0,
              288
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        },
        {
          "region_id": "R2",
          "name": "across_roundabout",
          "points_normalized": [
            [
              0.114286,
              0.350617
            ],
            [
              0.192857,
              0.332275
            ],
            [
              0.528571,
              0.37037
            ],
            [
              0.635714,
              0.392945
            ],
            [
              0.822222,
              0.409877
            ],
            [
              0.938889,
              0.593298
            ],
            [
              0.740234,
              0.572917
            ],
            [
              0.552734,
              0.572917
            ],
            [
              0.458984,
              0.576389
            ],
            [
              0.359375,
              0.510417
            ]
          ],
          "points_px": [
            [
              58.51,
              100.98
            ],
            [
              98.74,
              95.7
            ],
            [
              270.63,
              106.67
            ],
            [
              325.49,
              113.17
            ],
            [
              420.98,
              118.04
            ],
            [
              480.71,
              170.87
            ],
            [
              379,
              165
            ],
            [
              283,
              165
            ],
            [
              235,
              166
            ],
            [
              184,
              147
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        }
      ],
      "note": "Loại đảo cây/đài trang trí lớn bên phải. Chỉ giữ dải mặt đường quan sát được quanh đảo; vùng tối cần kiểm tra bằng ảnh ngày. Biên hai vùng sát nhau, cộng theo union thay vì cộng box hai lần."
    },
    {
      "index": 14,
      "camera_id": "5d8cdc57766c88001718896e",
      "name": "Ly_Thuong_Kiet_Lac_Long_Quan",
      "reference_image": "data/raw/snapshots/5d8cdc57766c88001718896e/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "183054c2904bf9dcd56a7d48b813e351ebd67462219048b0f16f0481e9aaf449",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "near_junction",
          "points_normalized": [
            [
              0,
              0.496528
            ],
            [
              0.4,
              0.425397
            ],
            [
              0.44127,
              0.44515
            ],
            [
              0.55,
              0.440917
            ],
            [
              0.656349,
              0.432452
            ],
            [
              0.742188,
              0.496528
            ],
            [
              0.833984,
              0.600694
            ],
            [
              0.923828,
              0.8125
            ],
            [
              0.970703,
              1
            ],
            [
              0.613281,
              1
            ],
            [
              0.625,
              0.881944
            ],
            [
              0.357422,
              0.857639
            ],
            [
              0.333333,
              0.862787
            ],
            [
              0.310547,
              0.864583
            ],
            [
              0.296875,
              1
            ],
            [
              0,
              1
            ]
          ],
          "points_px": [
            [
              0,
              143
            ],
            [
              204.8,
              122.51
            ],
            [
              225.93,
              128.2
            ],
            [
              281.6,
              126.98
            ],
            [
              336.05,
              124.55
            ],
            [
              380,
              143
            ],
            [
              427,
              173
            ],
            [
              473,
              234
            ],
            [
              497,
              288
            ],
            [
              314,
              288
            ],
            [
              320,
              254
            ],
            [
              183,
              247
            ],
            [
              170.67,
              248.48
            ],
            [
              159,
              249
            ],
            [
              152,
              288
            ],
            [
              0,
              288
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        },
        {
          "region_id": "R2",
          "name": "left_approach",
          "points_normalized": [
            [
              0.053175,
              0.416931
            ],
            [
              0.107143,
              0.350617
            ],
            [
              0.184921,
              0.241975
            ],
            [
              0.300781,
              0.239583
            ],
            [
              0.438889,
              0.220811
            ],
            [
              0.509524,
              0.316755
            ],
            [
              0.428571,
              0.412698
            ],
            [
              0.292969,
              0.444444
            ],
            [
              0,
              0.482639
            ]
          ],
          "points_px": [
            [
              27.23,
              120.08
            ],
            [
              54.86,
              100.98
            ],
            [
              94.68,
              69.69
            ],
            [
              154,
              69
            ],
            [
              224.71,
              63.59
            ],
            [
              260.88,
              91.23
            ],
            [
              219.43,
              118.86
            ],
            [
              150,
              128
            ],
            [
              0,
              139
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        },
        {
          "region_id": "R3",
          "name": "right_approach",
          "points_normalized": [
            [
              0.542857,
              0.333686
            ],
            [
              0.454762,
              0.223633
            ],
            [
              0.67381,
              0.194004
            ],
            [
              0.779297,
              0.197917
            ],
            [
              0.921875,
              0.177083
            ],
            [
              0.970703,
              0.222222
            ],
            [
              1,
              0.291667
            ],
            [
              1,
              0.984127
            ],
            [
              0.864286,
              0.638448
            ],
            [
              0.816667,
              0.565079
            ],
            [
              0.753906,
              0.482639
            ],
            [
              0.673828,
              0.454861
            ]
          ],
          "points_px": [
            [
              277.94,
              96.1
            ],
            [
              232.84,
              64.41
            ],
            [
              344.99,
              55.87
            ],
            [
              399,
              57
            ],
            [
              472,
              51
            ],
            [
              497,
              64
            ],
            [
              512,
              84
            ],
            [
              512,
              283.43
            ],
            [
              442.51,
              183.87
            ],
            [
              418.13,
              162.74
            ],
            [
              386,
              139
            ],
            [
              345,
              131
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        }
      ],
      "note": "Ba vùng tránh đảo tròn ở giữa và đảo giao thông tiền cảnh. Có nhiều hướng tại nút giao; không coi mỗi vùng là lane có hướng xác định. Ranh giới gần xe bus và quanh đảo cần duyệt."
    },
    {
      "index": 15,
      "camera_id": "5deb576d1dc17d7c5515ad08",
      "name": "Cong_Hoa_Tan_Ky_Tan_Quy",
      "reference_image": "data/raw/snapshots/5deb576d1dc17d7c5515ad08/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "24befb1e3fddb06acbd73eecf522892061c30a647ff73c556579a4d13e8f6ee5",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "whole_visible_road",
          "points_normalized": [
            [
              0,
              0.46875
            ],
            [
              0.09375,
              0.368056
            ],
            [
              0.191406,
              0.284722
            ],
            [
              0.275391,
              0.236111
            ],
            [
              0.547619,
              0.217989
            ],
            [
              0.65873,
              0.209524
            ],
            [
              0.759524,
              0.367549
            ],
            [
              0.849206,
              0.491711
            ],
            [
              1,
              0.604167
            ],
            [
              1,
              1
            ],
            [
              0,
              1
            ]
          ],
          "points_px": [
            [
              0,
              135
            ],
            [
              48,
              106
            ],
            [
              98,
              82
            ],
            [
              141,
              68
            ],
            [
              280.38,
              62.78
            ],
            [
              337.27,
              60.34
            ],
            [
              388.88,
              105.85
            ],
            [
              434.79,
              141.61
            ],
            [
              512,
              174
            ],
            [
              512,
              288
            ],
            [
              0,
              288
            ]
          ],
          "direction_note": "Gộp mặt đường nhìn thấy vì không có dải phân cách vật lý; chưa xác nhận hướng lưu thông."
        }
      ],
      "note": "Một ROI gộp mặt đường gần/trung cảnh, không chừa khe nhân tạo theo tim đường để tránh bỏ xe chuyển làn. Loại đoàn xe nhỏ phía rất xa. Node đại diện tổng phương tiện quan sát được trong vùng, chưa tách chiều."
    },
    {
      "index": 16,
      "camera_id": "5deb576d1dc17d7c5515ad09",
      "name": "Truong_Chinh_Hoang_Hoa_Tham",
      "reference_image": "data/raw/snapshots/5deb576d1dc17d7c5515ad09/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "e728133b634b019fb73aa9d5d4f138030e2681f13b2cf8fdbdb733227e41e98b",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "near_junction",
          "points_normalized": [
            [
              0,
              0.354167
            ],
            [
              0.136719,
              0.315972
            ],
            [
              0.236328,
              0.295139
            ],
            [
              0.375,
              0.267361
            ],
            [
              0.447266,
              0.256944
            ],
            [
              0.490476,
              0.226455
            ],
            [
              0.596032,
              0.154497
            ],
            [
              0.794444,
              0.179894
            ],
            [
              0.947619,
              0.239153
            ],
            [
              0.958984,
              0.503472
            ],
            [
              0.976562,
              1
            ],
            [
              0,
              1
            ]
          ],
          "points_px": [
            [
              0,
              102
            ],
            [
              70,
              91
            ],
            [
              121,
              85
            ],
            [
              192,
              77
            ],
            [
              229,
              74
            ],
            [
              251.12,
              65.22
            ],
            [
              305.17,
              44.5
            ],
            [
              406.76,
              51.81
            ],
            [
              485.18,
              68.88
            ],
            [
              491,
              145
            ],
            [
              500,
              288
            ],
            [
              0,
              288
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        }
      ],
      "note": "Giữ phần nút giao gần/trung cảnh; loại vùng xa lóa, dải phân cách phía trên phải và vỉa hè. ROI gồm nhiều chuyển động tại giao lộ, cần xác nhận phù hợp định nghĩa node."
    },
    {
      "index": 17,
      "camera_id": "5deb576d1dc17d7c5515ad0a",
      "name": "Hoang_Van_Thu_Xuan_Dieu",
      "reference_image": "data/raw/snapshots/5deb576d1dc17d7c5515ad0a/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "0019c21b084ed38147bf74f500719c99233460393bd0be196a977fea8ad1927a",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "left_of_median",
          "points_normalized": [
            [
              0.015625,
              1
            ],
            [
              0.153968,
              0.738624
            ],
            [
              0.230952,
              0.552381
            ],
            [
              0.296032,
              0.423986
            ],
            [
              0.330952,
              0.332275
            ],
            [
              0.409524,
              0.329453
            ],
            [
              0.507143,
              1
            ]
          ],
          "points_px": [
            [
              8,
              288
            ],
            [
              78.83,
              212.72
            ],
            [
              118.25,
              159.09
            ],
            [
              151.57,
              122.11
            ],
            [
              169.45,
              95.7
            ],
            [
              209.68,
              94.88
            ],
            [
              259.66,
              288
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        },
        {
          "region_id": "R2",
          "name": "right_of_median",
          "points_normalized": [
            [
              0.412109,
              0.315972
            ],
            [
              0.484921,
              0.318166
            ],
            [
              0.548828,
              0.423611
            ],
            [
              0.6875,
              0.586806
            ],
            [
              0.876953,
              0.84375
            ],
            [
              0.958984,
              1
            ],
            [
              0.521484,
              1
            ]
          ],
          "points_px": [
            [
              211,
              91
            ],
            [
              248.28,
              91.63
            ],
            [
              281,
              122
            ],
            [
              352,
              169
            ],
            [
              449,
              243
            ],
            [
              491,
              288
            ],
            [
              267,
              288
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        }
      ],
      "note": "Hai phần đường theo dải phân cách. Không lấy đường chân trời lóa đèn; phải kiểm tra mép đường trái bằng ảnh ngày. Xe lớn che xe nhỏ là giới hạn quan sát, không đổi ROI theo từng frame."
    },
    {
      "index": 18,
      "camera_id": "662b57471afb9c00172d9095",
      "name": "Cong_Hoa_Ap_Bac",
      "reference_image": "data/raw/snapshots/662b57471afb9c00172d9095/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "f8f283a018b227be47196b3bee619c59b90d2a74f33501754cc67dd02a6d3eb8",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "near_left_road",
          "points_normalized": [
            [
              0,
              0.425397
            ],
            [
              0.175397,
              0.3097
            ],
            [
              0.303175,
              0.251852
            ],
            [
              0.421429,
              0.2194
            ],
            [
              0.47619,
              0.225044
            ],
            [
              0.559524,
              0.253263
            ],
            [
              0.627778,
              0.263139
            ],
            [
              0.661905,
              0.274427
            ],
            [
              0.568359,
              0.458333
            ],
            [
              0.551587,
              0.480423
            ],
            [
              0.529297,
              0.569444
            ],
            [
              0.212891,
              1
            ],
            [
              0,
              1
            ]
          ],
          "points_px": [
            [
              0,
              122.51
            ],
            [
              89.8,
              89.19
            ],
            [
              155.23,
              72.53
            ],
            [
              215.77,
              63.19
            ],
            [
              243.81,
              64.81
            ],
            [
              286.48,
              72.94
            ],
            [
              321.42,
              75.78
            ],
            [
              338.9,
              79.03
            ],
            [
              291,
              132
            ],
            [
              282.41,
              138.36
            ],
            [
              271,
              164
            ],
            [
              109,
              288
            ],
            [
              0,
              288
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        },
        {
          "region_id": "R2",
          "name": "near_right_road",
          "points_normalized": [
            [
              0.671875,
              0.284722
            ],
            [
              0.742857,
              0.178483
            ],
            [
              0.851587,
              0.202469
            ],
            [
              0.810547,
              0.475694
            ],
            [
              0.785156,
              0.611111
            ],
            [
              0.794922,
              0.822917
            ],
            [
              0.78125,
              1
            ],
            [
              0.279297,
              1
            ],
            [
              0.552734,
              0.604167
            ],
            [
              0.560547,
              0.496528
            ],
            [
              0.597656,
              0.413194
            ]
          ],
          "points_px": [
            [
              344,
              82
            ],
            [
              380.34,
              51.4
            ],
            [
              436.01,
              58.31
            ],
            [
              415,
              137
            ],
            [
              402,
              176
            ],
            [
              407,
              237
            ],
            [
              400,
              288
            ],
            [
              143,
              288
            ],
            [
              283,
              174
            ],
            [
              287,
              143
            ],
            [
              306,
              119
            ]
          ],
          "direction_note": "Vùng hình học theo ảnh; chưa xác nhận hướng giao thông thực tế."
        }
      ],
      "note": "Hai vùng phần đường gần, loại dải phân cách, cây che bên phải và dải xe xa phía trên. Giới hạn quan sát không bao phủ toàn bộ giao lộ; kiểm tra lại các nhánh sau khi có ảnh ngày."
    },
    {
      "index": 19,
      "camera_id": "662b57711afb9c00172d90a7",
      "name": "Cong_Hoa_Duong_C4",
      "reference_image": "data/raw/snapshots/662b57711afb9c00172d90a7/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "c33de0b5fa903b2627a9e730ca237ad8fc24bac6703596610ccf502196b3b6e9",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "phan_duong_trai",
          "points_normalized": [
            [
              0.089844,
              0.302083
            ],
            [
              0.215079,
              0.2194
            ],
            [
              0.773438,
              0.979167
            ],
            [
              0.03125,
              0.979167
            ],
            [
              0.039062,
              0.604167
            ]
          ],
          "points_px": [
            [
              46,
              87
            ],
            [
              110.12,
              63.19
            ],
            [
              396,
              282
            ],
            [
              16,
              282
            ],
            [
              20,
              174
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        },
        {
          "region_id": "R2",
          "name": "phan_duong_phai",
          "points_normalized": [
            [
              0.271429,
              0.277249
            ],
            [
              0.415873,
              0.237743
            ],
            [
              0.995238,
              0.510053
            ],
            [
              0.990234,
              0.979167
            ],
            [
              0.814286,
              0.991182
            ],
            [
              0.543651,
              0.620106
            ]
          ],
          "points_px": [
            [
              138.97,
              79.85
            ],
            [
              212.93,
              68.47
            ],
            [
              509.56,
              146.9
            ],
            [
              507,
              282
            ],
            [
              416.91,
              285.46
            ],
            [
              278.35,
              178.59
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        }
      ],
      "note": "Tên trên ảnh là Cộng Hòa - Đường A4, trong danh sách là C4; cần xác nhận metadata. Loại vùng xa phía trên và dải phân cách; ảnh có đốm lóa/mờ bên trái. Lưu ý metadata: tên crawler ghi Đường C4 nhưng chữ trên ảnh ghi Đường A4; giữ camera ID làm định danh."
    },
    {
      "index": 20,
      "camera_id": "662b58401afb9c00172d90f5",
      "name": "Hoang_Van_Thu_Bui_Thi_Xuan",
      "reference_image": "data/raw/snapshots/662b58401afb9c00172d90f5/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "241d002152cd72a837720f871eea9c25cdf25a1ccbd008a15b512ed7fd04afeb",
      "review_status": "approved",
      "confidence": "low",
      "regions": [
        {
          "region_id": "R1",
          "name": "duong_mat_dat",
          "points_normalized": [
            [
              0,
              0.267372
            ],
            [
              0.210317,
              0.280071
            ],
            [
              0.291016,
              0.416667
            ],
            [
              0.420635,
              0.70194
            ],
            [
              0.525397,
              0.979894
            ],
            [
              0.105469,
              0.982639
            ]
          ],
          "points_px": [
            [
              0,
              77
            ],
            [
              107.68,
              80.66
            ],
            [
              149,
              120
            ],
            [
              215.37,
              202.16
            ],
            [
              269,
              282.21
            ],
            [
              54,
              283
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        },
        {
          "region_id": "R2",
          "name": "mat_duong_cau_vuot",
          "points_normalized": [
            [
              0.281746,
              0.148854
            ],
            [
              0.360317,
              0.072663
            ],
            [
              0.996032,
              0.298413
            ],
            [
              0.977778,
              0.876896
            ],
            [
              0.469841,
              0.366138
            ]
          ],
          "points_px": [
            [
              144.25,
              42.87
            ],
            [
              184.48,
              20.93
            ],
            [
              509.97,
              85.94
            ],
            [
              500.62,
              252.55
            ],
            [
              240.56,
              105.45
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        }
      ],
      "note": "Hai tầng đường khác nhau: mặt đất và cầu vượt. Cần quyết định tách thành hai tín hiệu hoặc tổng hai ROI; lan can che chân xe trên cầu làm quy tắc đáy box kém ổn định."
    },
    {
      "index": 21,
      "camera_id": "662b58791afb9c00172d9107",
      "name": "225_Hoang_Van_Thu",
      "reference_image": "data/raw/snapshots/662b58791afb9c00172d9107/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "620e552d542c18670a937297f67af04ac318ba8d3a1df5a4f8690d9cfba523ad",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "long_duong_quan_sat",
          "points_normalized": [
            [
              0.157143,
              0.281482
            ],
            [
              0.434127,
              0.239153
            ],
            [
              0.992188,
              0.565972
            ],
            [
              0.992188,
              0.982639
            ],
            [
              0.154297,
              0.982639
            ],
            [
              0.168254,
              0.71746
            ],
            [
              0.11746,
              0.449383
            ]
          ],
          "points_px": [
            [
              80.46,
              81.07
            ],
            [
              222.27,
              68.88
            ],
            [
              508,
              163
            ],
            [
              508,
              283
            ],
            [
              79,
              283
            ],
            [
              86.15,
              206.63
            ],
            [
              60.14,
              129.42
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        }
      ],
      "note": "Ảnh ghi Hoàng Văn Thụ - Số 255, tên cấu hình 225_Hoang_Van_Thu. Bỏ vùng xa và thân cây/vỉa hè bên trái; cành cây vẫn che một phần ảnh. Lưu ý metadata: tên crawler ghi 225 nhưng chữ trên ảnh ghi Số 255 Hoàng Văn Thụ; giữ camera ID làm định danh."
    },
    {
      "index": 22,
      "camera_id": "662b5a401afb9c00172d91fc",
      "name": "Truong_Chinh_Nguyen_Hong_Dao",
      "reference_image": "data/raw/snapshots/662b5a401afb9c00172d91fc/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "e12bd896383f1442f8dec19c93f12d0a2abf7c16522b3b984438a4a53078f028",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "phan_duong_trai",
          "points_normalized": [
            [
              0.009766,
              0.392361
            ],
            [
              0.451172,
              0.315972
            ],
            [
              0.738889,
              0.198236
            ],
            [
              0.852381,
              0.217989
            ],
            [
              0.55,
              0.718871
            ],
            [
              0.347619,
              1
            ],
            [
              0.009766,
              0.982639
            ]
          ],
          "points_px": [
            [
              5,
              113
            ],
            [
              231,
              91
            ],
            [
              378.31,
              57.09
            ],
            [
              436.42,
              62.78
            ],
            [
              281.6,
              207.03
            ],
            [
              177.98,
              288
            ],
            [
              5,
              283
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        },
        {
          "region_id": "R2",
          "name": "phan_duong_phai",
          "points_normalized": [
            [
              0.895238,
              0.202469
            ],
            [
              1,
              0.212346
            ],
            [
              0.992188,
              0.982639
            ],
            [
              0.517578,
              0.982639
            ],
            [
              0.708984,
              0.597222
            ]
          ],
          "points_px": [
            [
              458.36,
              58.31
            ],
            [
              512,
              61.16
            ],
            [
              508,
              283
            ],
            [
              265,
              283
            ],
            [
              363,
              172
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        }
      ],
      "note": "Tách hai phần đường theo dải phân cách chéo. Phần xa chỉ lấy tới khu vực xe còn phân biệt được; cần xem ban ngày để chốt mép trái sát công trình."
    },
    {
      "index": 23,
      "camera_id": "662b5a9c1afb9c00172d9240",
      "name": "Truong_Chinh_Nguyen_Thai_Binh",
      "reference_image": "data/raw/snapshots/662b5a9c1afb9c00172d9240/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "e14a78a73fb1a03d9714b253d02c390c7fc660b5c0eebbaff552138e29cccfb9",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "phan_duong_trai",
          "points_normalized": [
            [
              0.313492,
              0.201058
            ],
            [
              0.411111,
              0.191182
            ],
            [
              0.470703,
              0.364583
            ],
            [
              0.746825,
              0.975661
            ],
            [
              0.220703,
              0.982639
            ],
            [
              0.273438,
              0.572917
            ]
          ],
          "points_px": [
            [
              160.51,
              57.9
            ],
            [
              210.49,
              55.06
            ],
            [
              241,
              105
            ],
            [
              382.37,
              280.99
            ],
            [
              113,
              283
            ],
            [
              140,
              165
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        },
        {
          "region_id": "R2",
          "name": "phan_duong_phai",
          "points_normalized": [
            [
              0.42619,
              0.184127
            ],
            [
              0.51746,
              0.165785
            ],
            [
              0.992188,
              0.600694
            ],
            [
              0.992188,
              0.982639
            ],
            [
              0.801587,
              0.991182
            ],
            [
              0.542969,
              0.447917
            ]
          ],
          "points_px": [
            [
              218.21,
              53.03
            ],
            [
              264.94,
              47.75
            ],
            [
              508,
              173
            ],
            [
              508,
              283
            ],
            [
              410.41,
              285.46
            ],
            [
              278,
              129
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        }
      ],
      "note": "Loại dải phân cách và vỉa hè; cây bên trái có thể che xe. Bỏ vùng đèn chói xa phía trên."
    },
    {
      "index": 24,
      "camera_id": "662b5b271afb9c00172d9296",
      "name": "Truong_Chinh_Tan_Hai",
      "reference_image": "data/raw/snapshots/662b5b271afb9c00172d9296/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "97a3fbe7a885c6123b678ea4170ffd805e9f6c32e54cd60efdfd3c9b60d837d8",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "phan_duong_trai",
          "points_normalized": [
            [
              0.246032,
              0.24903
            ],
            [
              0.324603,
              0.239153
            ],
            [
              0.402344,
              0.4375
            ],
            [
              0.662698,
              0.989771
            ],
            [
              0.21875,
              0.982639
            ],
            [
              0.228516,
              0.625
            ]
          ],
          "points_px": [
            [
              125.97,
              71.72
            ],
            [
              166.2,
              68.88
            ],
            [
              206,
              126
            ],
            [
              339.3,
              285.05
            ],
            [
              112,
              283
            ],
            [
              117,
              180
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        },
        {
          "region_id": "R2",
          "name": "phan_duong_phai",
          "points_normalized": [
            [
              0.360317,
              0.260317
            ],
            [
              0.471429,
              0.237743
            ],
            [
              0.730469,
              0.5625
            ],
            [
              0.992188,
              0.774306
            ],
            [
              0.992188,
              0.982639
            ],
            [
              0.679688,
              0.982639
            ],
            [
              0.466797,
              0.496528
            ]
          ],
          "points_px": [
            [
              184.48,
              74.97
            ],
            [
              241.37,
              68.47
            ],
            [
              374,
              162
            ],
            [
              508,
              223
            ],
            [
              508,
              283
            ],
            [
              348,
              283
            ],
            [
              239,
              143
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        }
      ],
      "note": "Hai phần đường ngăn bởi hàng rào/dải phân cách; mép xa và vỉa hè bị cây che. Cần ảnh ngày xác nhận mép làn sát lề phải."
    },
    {
      "index": 25,
      "camera_id": "662b5b481afb9c00172d92a8",
      "name": "Truong_Chinh_Truong_Cong_Dinh",
      "reference_image": "data/raw/snapshots/662b5b481afb9c00172d92a8/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "796cb9cb4d81911dd3828ecc5f74d7fc474efe96339a7ad1d3c114a7387ec9a2",
      "review_status": "approved",
      "confidence": "low",
      "regions": [
        {
          "region_id": "R1",
          "name": "phan_duong_gan",
          "points_normalized": [
            [
              0.019531,
              0.381944
            ],
            [
              0.079365,
              0.352028
            ],
            [
              0.760317,
              1
            ],
            [
              0.150391,
              0.982639
            ],
            [
              0.023016,
              0.483245
            ]
          ],
          "points_px": [
            [
              10,
              110
            ],
            [
              40.63,
              101.38
            ],
            [
              389.28,
              288
            ],
            [
              77,
              283
            ],
            [
              11.78,
              139.17
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        },
        {
          "region_id": "R2",
          "name": "phan_duong_xa",
          "points_normalized": [
            [
              0.139683,
              0.356261
            ],
            [
              0.237302,
              0.332275
            ],
            [
              0.992188,
              0.59375
            ],
            [
              0.993651,
              0.968607
            ],
            [
              0.541016,
              0.628472
            ],
            [
              0.238095,
              0.438095
            ]
          ],
          "points_px": [
            [
              71.52,
              102.6
            ],
            [
              121.5,
              95.7
            ],
            [
              508,
              171
            ],
            [
              508.75,
              278.96
            ],
            [
              277,
              181
            ],
            [
              121.9,
              126.17
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        }
      ],
      "note": "Dải phân cách chéo chiếm nhiều khung hình. Loại đoạn xa bên trái và công trình phía trên; nhãn xe xa ban đêm khó kiểm chứng. Mép ROI trên phần đường xa cần kiểm tra bằng ảnh ngày."
    },
    {
      "index": 26,
      "camera_id": "66b1c4a2779f74001867418c",
      "name": "Le_Van_Si_Pham_Van_Hai",
      "reference_image": "data/raw/snapshots/66b1c4a2779f74001867418c/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "22d3ab8a6edb75d7e615b205ee4177cfca52221cd3d47dbec0efb959192eceb4",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "doan_duong_truoc_nut_giao",
          "points_normalized": [
            [
              0.335714,
              0.171429
            ],
            [
              0.557143,
              0.185538
            ],
            [
              0.611111,
              0.171429
            ],
            [
              0.625397,
              0.215168
            ],
            [
              0.603968,
              0.29418
            ],
            [
              1,
              0.985538
            ],
            [
              0.298828,
              0.979167
            ]
          ],
          "points_px": [
            [
              171.89,
              49.37
            ],
            [
              285.26,
              53.43
            ],
            [
              312.89,
              49.37
            ],
            [
              320.2,
              61.97
            ],
            [
              309.23,
              84.72
            ],
            [
              512,
              283.83
            ],
            [
              153,
              282
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        }
      ],
      "note": "Lấy đoạn đường trước vạch qua đường, gồm hai phía quan sát trong cùng một ROI; chưa lấy vùng giao cắt xa để giảm nhập nhằng. Loại vỉa hè/cây và xe đỗ ngoài lòng đường."
    },
    {
      "index": 27,
      "camera_id": "66b1c4e7779f7400186741e4",
      "name": "Hoang_Van_Thu_Pham_Van_Hai",
      "reference_image": "data/raw/snapshots/66b1c4e7779f7400186741e4/20261114_214301.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "a2d6562a1e33701446081bf25ba06ba40d97eb174e0e6e208f5daa5bef9b71ea",
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "phan_duong_trai",
          "points_normalized": [
            [
              0.411111,
              0.238801
            ],
            [
              0.496825,
              0.251499
            ],
            [
              0.490234,
              0.986111
            ],
            [
              0.028571,
              0.995062
            ],
            [
              0.152381,
              0.776367
            ],
            [
              0.257143,
              0.546384
            ]
          ],
          "points_px": [
            [
              210.49,
              68.77
            ],
            [
              254.37,
              72.43
            ],
            [
              251,
              284
            ],
            [
              14.63,
              286.58
            ],
            [
              78.02,
              223.59
            ],
            [
              131.66,
              157.36
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        },
        {
          "region_id": "R2",
          "name": "phan_duong_phai",
          "points_normalized": [
            [
              0.503968,
              0.25291
            ],
            [
              0.615079,
              0.257143
            ],
            [
              0.672222,
              0.446208
            ],
            [
              0.748413,
              0.635273
            ],
            [
              0.896032,
              0.99224
            ],
            [
              0.509766,
              0.982639
            ]
          ],
          "points_px": [
            [
              258.03,
              72.84
            ],
            [
              314.92,
              74.06
            ],
            [
              344.18,
              128.51
            ],
            [
              383.19,
              182.96
            ],
            [
              458.77,
              285.77
            ],
            [
              261,
              283
            ]
          ],
          "direction_note": "Hướng mô tả theo vị trí trong ảnh; cần kiểm tra hướng lưu thông bằng chuỗi ảnh."
        }
      ],
      "note": "Tách hai phần đường quan sát theo trục giữa ảnh; chưa xác nhận hướng lưu thông thực tế. Cây che nhiều phía phải; không thể loại hết cành che bằng đa giác đơn, cần đánh dấu chất lượng quan sát."
    },
    {
      "index": 28,
      "camera_id": "5d8cdb4b766c880017188966",
      "name": "Hoang_Van_Thu_Ut_Tich",
      "reference_image": "data/raw/snapshots/5d8cdb4b766c880017188966/20261114_104306.jpg",
      "reference_size": [
        512,
        288
      ],
      "reference_image_sha256": "db8f9492f5fab2f13a09eb9dc8a79a3225da0d021d12e5be981638c31beaafce",
      "reference_source": {
        "repo_id": "ericng17/hcm_traffic_snapshots5",
        "path": "snapshots/5d8cdb4b766c880017188966/20261114_104306.jpg",
        "matches_user_attachment": true
      },
      "review_status": "approved",
      "confidence": "medium",
      "regions": [
        {
          "region_id": "R1",
          "name": "junction_excluding_utility_cabinet",
          "points_normalized": [
            [
              0.035156,
              0.236111
            ],
            [
              0.109375,
              0.197917
            ],
            [
              0.204762,
              0.188007
            ],
            [
              0.292064,
              0.20776
            ],
            [
              0.37619,
              0.278307
            ],
            [
              0.476562,
              0.357639
            ],
            [
              0.570312,
              0.385417
            ],
            [
              0.654297,
              0.388889
            ],
            [
              0.775397,
              0.4194
            ],
            [
              0.888672,
              0.409722
            ],
            [
              0.996094,
              0.444444
            ],
            [
              0.996094,
              0.993056
            ],
            [
              0.310317,
              0.996473
            ],
            [
              0.311905,
              0.896296
            ],
            [
              0.28254,
              0.664903
            ],
            [
              0.085714,
              0.70582
            ],
            [
              0.052734,
              0.506944
            ]
          ],
          "points_px": [
            [
              18,
              68
            ],
            [
              56,
              57
            ],
            [
              104.84,
              54.15
            ],
            [
              149.54,
              59.83
            ],
            [
              192.61,
              80.15
            ],
            [
              244,
              103
            ],
            [
              292,
              111
            ],
            [
              335,
              112
            ],
            [
              397,
              120.79
            ],
            [
              455,
              118
            ],
            [
              510,
              128
            ],
            [
              510,
              286
            ],
            [
              158.88,
              286.98
            ],
            [
              159.7,
              258.13
            ],
            [
              144.66,
              191.49
            ],
            [
              43.89,
              203.28
            ],
            [
              27,
              146
            ]
          ],
          "direction_note": "Giữ phần đường gần và nút giao; chưa phân tách hướng lưu thông từ một ảnh."
        }
      ],
      "note": "ROI lõm loại tủ điện lớn ở góc dưới trái, vỉa hè Co.opmart và phần đường quá xa phía trên trái. Giữ xe đang chờ/đi ngang/chuyển hướng trong vùng; phần mặt đường bị tủ che không quan sát được. Ảnh ngày 14/11/2026 lúc 10:43:05 trên overlay (tên file crawler 10:43:06), cần kiểm tra thêm ảnh đêm và góc nhìn ổn định trước khi duyệt."
    }
  ],
  "missing_cameras": []
}
'''

ROI_PATH = WORKDIR / 'camera_rois.reviewed.json'
ROI_PATH.write_text(ROI_JSON, encoding='utf-8')
rois, roi_hash, _ = load_rois(ROI_PATH)
print(f'ROI hợp lệ: {len(rois)} camera | SHA256: {roi_hash}')


## 5. Đọc weights và kiểm tra class

Cell đọc `.pt` trong máy chạy kernel. Mapping v20 dự kiến: 0 = xe hai bánh,
1 = bus, 2 = car, 3 = truck. Xem mapping thực tế và sửa `VEHICLE_CLASSES` nếu cần.


In [ ]:
if not WEIGHTS.is_file():
    raise FileNotFoundError(f'Không có weights: {WEIGHTS}. Kiểm tra WEIGHTS_PATH trên máy chạy kernel.')

from ultralytics import YOLO
check_model = YOLO(str(WEIGHTS))
try:
    assert check_model.task == 'detect', 'Cần weights detection.'
    model_names = {int(k): v for k, v in check_model.names.items()}
    print('Model mapping:', model_names)
    assert VEHICLE_CLASSES and set(VEHICLE_CLASSES) <= model_names.keys(), 'Sửa VEHICLE_CLASSES theo mapping.'
    print('Các class sẽ đếm:', {k: model_names[k] for k in sorted(set(VEHICLE_CLASSES))})
finally:
    del check_model
    torch.cuda.empty_cache()
rois, roi_hash, _ = load_rois(ROI_PATH)
print('Weights:', WEIGHTS)
print('ROI đã duyệt:', len(rois), 'camera')


## 6. Chạy lần lượt 5 dataset

Mặc định `LIMIT = 0` để xử lý hết dữ liệu trong `HF_REPOS` theo thứ tự đã hiển thị.
Log ghi tên và revision khi chuyển dataset. Cùng một model được dùng xuyên suốt
lần chạy; dữ liệu được tải từng batch ảnh hoặc một shard Parquet vào thư mục tạm.

Đặt `LIMIT = 1000` nếu muốn chạy thử trước. Sau khi xem preview, đặt lại `LIMIT = 0`,
chạy lại cell cấu hình và cell inference để tiếp tục. Giới hạn áp dụng cho tổng
số ảnh mới của lần gọi. Nếu OOM, giảm `BATCH` rồi chạy lại.

Khi `BACKUP_ENABLED = True`, worker nền sao lưu mỗi `BACKUP_INTERVAL_HOURS = 1` giờ.
SQLite Backup API tạo snapshot nhất quán từ các batch đã commit, xuất CSV của snapshot
và thay file checkpoint tại `CHECKPOINT_BACKUP` qua file tạm. Quá trình đếm vẫn tiếp tục trong lúc
sao lưu. Lần kế tiếp được tính sau khi lần sao lưu trước hoàn tất; log báo rõ khi
sao lưu thành công hoặc lỗi. Nếu lỗi, worker thử lại sau một chu kỳ.

Nút Stop vẫn cố gắng xuất CSV và sao lưu thêm một lần trong `finally`, sau khi đợi
worker nền hoàn tất. Nếu máy GPU bị kill cứng/mất runtime, resume từ lần sao lưu thành
công gần nhất; tiến độ mới hơn có thể phải chạy lại. Repo/camera/thời gian/ROI không hợp lệ
làm dừng run; sửa lỗi rồi resume thay vì bỏ qua dataset.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True)
assert SOURCE_MODE in {'huggingface', 'local'}
assert LIMIT >= 0
if SOURCE_MODE == 'huggingface':
    print('Chạy lần lượt:', ' → '.join(HF_REPOS))
    print('Giới hạn ảnh mới chung:', LIMIT if LIMIT else 'không giới hạn')
argv = [
    '--weights', str(WEIGHTS), '--roi', str(ROI_PATH), '--output', str(OUTPUT),
    '--classes', *map(str, VEHICLE_CLASSES), '--device', '0',
    '--batch', str(BATCH), '--imgsz', str(IMGSZ), '--conf', str(CONF),
    '--iou', str(IOU), '--max-det', str(MAX_DET), '--timezone', TIMEZONE,
    '--preview-count', str(PREVIEW_COUNT), '--workers', str(WORKERS),
    '--format', HF_FORMAT, '--image-column', IMAGE_COLUMN,
]
if SOURCE_MODE == 'huggingface':
    argv += ['--hf-repos', *HF_REPOS]
else:
    argv += ['--source', str(LOCAL_SOURCE)]
for flag, value in [('--path-column', PATH_COLUMN), ('--camera-column', CAMERA_COLUMN),
                    ('--timestamp-column', TIMESTAMP_COLUMN)]:
    if value is not None:
        argv += [flag, value]
if LIMIT:
    argv += ['--limit', str(LIMIT)]
if HALF:
    argv.append('--half')
if RETRY_ERRORS:
    argv.append('--retry-errors')

periodic_backup = (PeriodicCheckpoint(OUTPUT, CHECKPOINT_BACKUP, BACKUP_INTERVAL_HOURS * 3600)
                   if BACKUP_ENABLED else None)
try:
    if periodic_backup is not None:
        periodic_backup.start()
        print(f'Tự động sao lưu mỗi {BACKUP_INTERVAL_HOURS:g} giờ tới {CHECKPOINT_BACKUP}.')
    run(parser().parse_args(argv))
finally:
    if periodic_backup is not None:
        periodic_backup.stop()
    torch.cuda.empty_cache()
    backup_output()
print('CSV:', OUTPUT / 'counts.csv')


## 7. Xem số đếm và ảnh preview

CSV có `total_vehicles`, các cột `count_<id>_<tên class>`, camera, timestamp, trạng
thái, hash ảnh và hash ROI. Ảnh hỏng để trống số đếm; ảnh hợp lệ không có xe trong ROI
có giá trị 0. Preview lấy các ảnh xử lý đầu tiên, không bảo đảm đủ mọi camera/ngày/đêm.

Đường xanh là ROI; khung/chấm vàng là xe được tính. Kiểm tra bằng mắt điểm giữa đáy
khung xe, xe trên biên và xe bị che. Ảnh đổi góc/crop vẫn cần duyệt lại ROI.

In [ ]:
from contextlib import closing
from IPython.display import display
import pandas as pd

assert (OUTPUT / 'counts.sqlite').is_file(), 'Chạy inference trước.'
with closing(sqlite3.connect(OUTPUT / 'counts.sqlite')) as db:
    display(pd.read_sql_query(
        'SELECT status, COUNT(*) AS images FROM counts GROUP BY status', db))
    display(pd.read_sql_query(
        'SELECT repo, status, COUNT(*) AS images FROM counts GROUP BY repo,status ORDER BY repo,status', db))
    display(pd.read_sql_query(
        'SELECT camera_id, COUNT(*) AS valid_images, AVG(total) AS mean_vehicles_in_roi, '
        'MAX(total) AS max_vehicles_in_roi FROM counts WHERE status="ok" GROUP BY camera_id', db))
    display(pd.read_sql_query(
        'SELECT camera_id, timestamp, status, total AS total_vehicles, counts AS counts_by_class, error '
        'FROM counts ORDER BY camera_id,timestamp LIMIT 20', db))

SHOW_PREVIEWS = 6 #@param {type:'integer'}
from itertools import islice
assert SHOW_PREVIEWS >= 0
for path in islice(sorted((OUTPUT / 'previews').glob('*.jpg')), SHOW_PREVIEWS):
    print(path.name)
    with Image.open(path) as preview:
        display(preview.copy())


## 8. Sao lưu và tải kết quả

Muốn resume ở phiên mới cần giữ toàn bộ snapshot, đặc biệt `counts.sqlite`,
`run.json`, `roi.json`. Cell này sao lưu thêm sau khi inference dừng. Colab dùng
hộp thoại download; Jupyter/VS Code cung cấp link và đường dẫn file trên server.
Để có bản trên laptop, dùng SFTP/rsync thư mục `CHECKPOINT_BACKUP` về máy bạn.
Chọn `all` để đóng ZIP; ZIP có thể lớn.


In [ ]:
from IPython.display import FileLink, display

DOWNLOAD = 'csv' #@param ['csv', 'all', 'none']
assert (OUTPUT / 'counts.sqlite').is_file(), 'Chưa có checkpoint inference.'
backup_output()
artifact = None
if DOWNLOAD == 'csv':
    artifact = OUTPUT / 'counts.csv'
    assert artifact.is_file(), 'Chưa có counts.csv.'
elif DOWNLOAD == 'all':
    artifact = Path(shutil.make_archive(str(WORKDIR / (RUN_NAME + '-results')), 'zip', OUTPUT))

if artifact is not None:
    if IS_COLAB:
        from google.colab import files
        files.download(str(artifact))
    else:
        print('File trên máy GPU:', artifact)
        display(FileLink(os.path.relpath(artifact)))
        print('Nếu link nằm ngoài thư mục Jupyter đang phục vụ, tải bằng trình duyệt file của server hoặc SFTP/rsync.')
print('Output:', OUTPUT)
if BACKUP_ENABLED:
    print('Checkpoint để đồng bộ về laptop:', CHECKPOINT_BACKUP)
